Download ERA5-Land climate data

In [ ]:
import ee
import geemap

In [ ]:
# Initialize the Earth Engine module.
# Chenge your own Google Cloud project ID
MY_PROJECT_ID = 'your-google-cloud-project-id'  
try:
    ee.Initialize(project=MY_PROJECT_ID)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT_ID)

print("GEE Initialized successfully with project ID:", MY_PROJECT_ID)

In [ ]:
# Area of interest and year
region = ee.Geometry.Rectangle([65, 24, 107, 48], 'EPSG:4326', False)
year = '2000'
# ERA5-Land Monthly Aggregated
dset = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
         .filter(ee.Filter.date(f'{year}-01-01', f'{year}-12-31'))
print(dset.size().getInfo())

In [ ]:
# Temperature processing (2m temperature, Kelvin K to Celsius °C)
imgcol_t2m = dset.select(['temperature_2m'])
t2m_yearly_mean = imgcol_t2m.mean().clip(region).subtract(273.15)

# Precipitation processing (cumulative precipitation, meters m to millimeters mm)
imgcol_tp = dset.select(['total_precipitation_sum'])
tp_yearly_sum = imgcol_tp.sum().clip(region).multiply(1000)

# Total evaporation processing (cumulative total evaporation, meters m to millimeters mm)
imgcol_te = dset.select(['total_evaporation_sum'])
te_yearly_sum = imgcol_te.sum().clip(region).multiply(1000)

In [ ]:
# visualization parameters
visual = {
    'min': 0.0,
    'max': 6000,
    'palette': [
        '000080', '0000d9', '4000ff', '8000ff', '0080ff', '00ffff',
        '00ff80', '80ff00', 'daff00', 'ffff00', 'fff500', 'ffda00',
        'ffb000', 'ffa400', 'ff4f00', 'ff2500', 'ff0a00', 'ff00ff',
    ]
}

# Area of interest outline
empty = ee.Image().byte()
scene_outline = empty.paint(
    featureCollection=region,
    color=1,
    width=3
)

In [ ]:
# Visualize the results using geemap
Map = geemap.Map()
Map.centerObject(region, 4)
# Map.addLayer(t2m_yearly_mean, visual, 'total tempreture')
Map.addLayer(tp_yearly_sum, visual, 'total precipitation')
# Map.addLayer(te_yearly_sum, visual, 'total evaporation')
Map.addLayer(scene_outline, {'palette': 'FF0000'}, 'training region')

# Show the map in Jupyter Notebook
Map


In [ ]:
# ==============================================================
# 10. 导出到 Google Drive
# ==============================================================
# projection = dset.first().projection().getInfo()
# task = ee.batch.Export.image.toDrive(
#     image=te_yearly_sum,
#     description=f'era5_land_yearly_te_{year}',
#     folder='tmp',
#     scale=11132,
#     crs=projection['crs'],
#     crsTransform=projection['transform'],
#     fileFormat='GeoTIFF',
#     region=region
# )
# task.start()
# print(f"导出任务已提交，Task ID: {task.id}")